# conv-windowing-1d composite — cx13: full 1D conv = windowed view contracted by einsum

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-windowing-1d`, `einops-einsum`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-windowing-1d"
DD_ATOM_IDS = ["conv-windowing-1d", "einops-einsum"]
DD_SUBTOPICS = ["CNN: 1-D conv windowing", "Einops: Deep Learning"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's from-scratch `conv1d` decomposes into **two atoms**:

1. **`conv-windowing-1d`** — turn input `x: (B, IC, W)` into a strided view `x_win: (B, IC, OW, KW)` where each `(KW,)` slice along the new `OW` axis is one kernel-sized window. Built via `x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, s_w, s_w))` — **no copy**.
2. **`einops-einsum`** — contract the window view against the kernel `weight: (OC, IC, KW)` along the shared `(ic, kw)` axes:
   ```
   einops.einsum(x_win, weight, 'b ic ow kw, oc ic kw -> b oc ow')
   ```

**Why this composition matters.** `F.conv1d` is a black box; this two-step decomposition is the ARENA *whitebox* form. Step 1 is pure view manipulation (free); step 2 is one fused tensor contraction. The result matches `F.conv1d(x, weight)` to fp tolerance — the proof that you understand convolution as 'sliding dot product'.

**Anatomy of the einsum pattern.** Free axes (`b oc ow`) appear on the RHS. Contracted axes (`ic kw`) appear on both LHS operands but NOT on the RHS — einsum sums over them. `oc` only appears on the kernel side; `b ow` only appear on the input side. Aligning the letters across the two operands is what makes the math match `conv1d`.

### Composite Exercise — full 1D conv = windowed view contracted by einsum

**Atoms exercised together**: `conv-windowing-1d`, `einops-einsum`

Implement `cx13_conv1d_full(x, weight)` — the whitebox replacement for `F.conv1d(x, weight)` (stride=1, no padding).

- `x`: float tensor of shape `(B, IC, W)`.
- `weight`: float tensor of shape `(OC, IC, KW)`.
- Return: tensor of shape `(B, OC, OW)` where `OW = W - KW + 1`.

1. **Window** — use `x.as_strided` to build `x_win` of shape `(B, IC, OW, KW)`. Read strides from `x.stride()` — do NOT hardcode.
2. **Einsum** — call `einops.einsum(x_win, weight, 'b ic ow kw, oc ic kw -> b oc ow')`.

The test compares your output against `F.conv1d(x, weight)` on multiple shapes.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx13_conv1d_full(x, weight):
    raise NotImplementedError

def _test_cx13():
    from torch.nn import functional as F
    rng = t.Generator().manual_seed(13)

    # Case A: tiny hand-checked example.
    x = t.arange(1.0, 11.0).reshape(1, 1, 10)
    w = t.tensor([[[1.0, 0.0, -1.0]]])  # (OC=1, IC=1, KW=3)
    y = cx13_conv1d_full(x, w)
    yref = F.conv1d(x, w)
    assert tuple(y.shape) == tuple(yref.shape), f'shape: {tuple(y.shape)} vs {tuple(yref.shape)}'
    assert t.allclose(y, yref, atol=1e-5), 'value mismatch on hand example'

    # Case B: multi-channel cross-check on several shapes.
    for B, IC, W, OC, KW in [(2,3,12,4,5),(1,1,8,1,3),(3,2,20,5,7),(2,4,16,8,1)]:
        x2 = t.randn(B, IC, W, generator=rng)
        w2 = t.randn(OC, IC, KW, generator=rng)
        yref = F.conv1d(x2, w2)
        yours = cx13_conv1d_full(x2, w2)
        assert tuple(yours.shape) == tuple(yref.shape)
        assert t.allclose(yours, yref, atol=1e-4), f'mismatch on B={B},IC={IC},W={W},OC={OC},KW={KW}'

    # Case C: edge — KW == W → single window per batch/channel.
    x3 = t.randn(2, 3, 7, generator=rng)
    w3 = t.randn(4, 3, 7, generator=rng)
    yours = cx13_conv1d_full(x3, w3)
    assert tuple(yours.shape) == (2, 4, 1)
    assert t.allclose(yours, F.conv1d(x3, w3), atol=1e-4)
    _dd_passed.add('cx13')

_test_cx13()

<details><summary>Show solution — cx13</summary>

```python
def cx13_conv1d_full(x, weight):
    B, IC, W = x.shape
    OC, IC2, KW = weight.shape
    assert IC == IC2, 'in_channels must match'
    OW = W - KW + 1
    # Atom A (conv-windowing-1d): build the (B, IC, OW, KW) strided view.
    s_b, s_ic, s_w = x.stride()
    x_win = x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, s_w, s_w))
    # Atom B (einops-einsum): contract over (ic, kw); keep (b, oc, ow).
    return einops.einsum(x_win, weight, 'b ic ow kw, oc ic kw -> b oc ow')
```

The two atoms ARE the implementation. There is no other code. Recognising this decomposition is what lets you generalize: add stride by multiplying `s_w * stride` on the `OW` axis, add padding by pre-padding `x` with zeros, add dilation by multiplying `s_w * dilation` on the `KW` axis. The einsum pattern stays unchanged.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx13'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx13',
        'subtopics': ["CNN: 1-D conv windowing", "Einops: Deep Learning"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()